[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/laboratorios/S16/S16_01_demostrador.ipynb)

# S16 · 01 — Plantilla de demostrador

**Tiempo estimado:** 1 h 15 min · **Sprint 16 (Proyecto: evaluación, IA responsable y entrega)** · píldora de apoyo

**Contexto.** La rúbrica pide un **demostrador ejecutable** (requisito R7) y que sea **reproducible desde el
README** (criterio F). Un demostrador no es "el notebook con todo": es una herramienta con la que **otra
persona** prueba tu modelo sin saber programar, con **avisos de incertidumbre** y **errores del usuario bien
gestionados**. Aquí construimos una plantilla completa con el modelo de cancelaciones de TurisData que tu equipo
adaptará a su proyecto.

**Cuatro opciones de demostrador (elige una, la primera es la recomendada):**

| Opción | Cuándo | Ventaja | Inconveniente |
|---|---|---|---|
| **Streamlit** (`app.py`) | Modelos tabulares, series, texto | Web interactiva con pocas líneas | Hay que instalarlo y lanzar `streamlit run` |
| **Gradio** | Imágenes, audio, chat (Hugging Face) | Muy rápido para subir una foto o escribir | Menos control del diseño |
| **Notebook con widgets** | Sin instalar nada / como plan B | Funciona en Colab | Menos "producto" |
| **API con FastAPI** (opcional) | Si otro sistema debe llamar al modelo | Estándar en producción | Requiere cliente que la consuma |

**Al terminar sabrás:**
1. Guardar un modelo con sus **metadatos** (columnas, rangos, versiones) y cargarlo desde otro archivo.
2. Separar la **lógica** (validar y predecir) de la **interfaz** (app, widgets, API).
3. Gestionar entradas erróneas del usuario y mostrar la **incertidumbre** del modelo.
4. Escribir **pruebas** de sentido común para el modelo.
5. Generar un `app.py` de Streamlit, un panel con widgets y (opcional) una API con FastAPI.

In [ ]:
# Preparación (ejecuta esta celda siempre la primera)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta


In [ ]:
import os, sys, json, py_compile
from datetime import date
import numpy as np
import pandas as pd
import joblib
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

SEMILLA = 42
np.random.seed(SEMILLA)
CARPETA = "demostrador"                    # aquí se crean modelo/, logica.py, app.py y api.py
os.makedirs(os.path.join(CARPETA, "modelo"), exist_ok=True)

## 1. Guardar el modelo y sus metadatos

**Idea.** El demostrador **no reentrena**: carga un modelo ya entrenado. Guardamos junto al modelo un
archivo `metadatos.json` con todo lo que la interfaz necesita saber: qué campos pedir, qué rangos son
válidos, qué categorías existen, con qué datos y versiones se entrenó y **cuánto vale el modelo**
(así el usuario ve sus límites). El **pipeline** (preprocesado + modelo) se guarda entero con `joblib`,
para que la app reciba una fila "cruda" igual que en el entrenamiento.

**Ejemplo resuelto:** entrenamos una regresión logística con 11 variables sencillas.

In [ ]:
reservas = pd.read_csv(dato("reservas_turisdata.csv"))
NUMERICAS = ["antelacion_dias", "noches", "adultos", "ninos", "precio_noche", "cliente_repetidor", "cancelaciones_previas"]
CATEGORICAS = ["tipo_habitacion", "canal", "tarifa", "deposito"]
X, y = reservas[NUMERICAS + CATEGORICAS], reservas["cancelada"]
X_ent, X_test, y_ent, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEMILLA)

prep = ColumnTransformer([("num", StandardScaler(), NUMERICAS), ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAS)])
modelo = Pipeline([("prep", prep), ("m", LogisticRegression(max_iter=2000))]).fit(X_ent, y_ent)
auc_test = roc_auc_score(y_test, modelo.predict_proba(X_test)[:, 1])
print(f"AUC en test: {auc_test:.3f}")

In [ ]:
meta = {
    "nombre": "Riesgo de cancelación de reservas · TurisData (datos sintéticos)",
    "campos": {
        "antelacion_dias": {"tipo": "entero", "min": int(X["antelacion_dias"].min()), "max": int(X["antelacion_dias"].max()), "etiqueta": "Antelación (días entre reserva y llegada)"},
        "noches": {"tipo": "entero", "min": 1, "max": 30, "etiqueta": "Noches"},
        "adultos": {"tipo": "entero", "min": 1, "max": 6, "etiqueta": "Adultos"},
        "ninos": {"tipo": "entero", "min": 0, "max": 6, "etiqueta": "Niños"},
        "precio_noche": {"tipo": "decimal", "min": 20.0, "max": 600.0, "etiqueta": "Precio por noche (€)"},
        "cliente_repetidor": {"tipo": "entero", "min": 0, "max": 1, "etiqueta": "Cliente repetidor (0 = no, 1 = sí)"},
        "cancelaciones_previas": {"tipo": "entero", "min": 0, "max": 10, "etiqueta": "Cancelaciones previas"},
        **{c: {"tipo": "categoria", "valores": sorted(X[c].unique().tolist()), "etiqueta": c.replace("_", " ").capitalize()} for c in CATEGORICAS},
    },
    "auc_test": round(float(auc_test), 3),
    "n_entrenamiento": int(len(X_ent)),
    "tasa_cancelacion_base": round(float(y.mean()), 3),
    "entrenado": str(date.today()),
    "semilla": SEMILLA,
    "versiones": {"python": sys.version.split()[0], "scikit-learn": sklearn.__version__, "pandas": pd.__version__, "numpy": np.__version__},
    "limitaciones": "Entrenado con datos sintéticos de un solo alojamiento ficticio. No usar para decidir sobre personas concretas: solo orienta la gestión de disponibilidad.",
}
joblib.dump(modelo, os.path.join(CARPETA, "modelo", "modelo_cancelaciones.joblib"))
with open(os.path.join(CARPETA, "modelo", "metadatos.json"), "w", encoding="utf8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)
print("Guardado:", os.listdir(os.path.join(CARPETA, "modelo")))

## 2. La lógica, separada de la interfaz

**Idea.** Si validar y predecir viven **dentro** de la app, no se pueden probar ni reutilizar en el
notebook o en la API. Ponemos esa lógica en un archivo `logica.py` **sin nada de Streamlit**. La app,
los widgets y la API serán capas finas que solo llaman a `logica.py`.

**Ejemplo resuelto:** escribimos `logica.py`. Léelo con calma: es lo que adaptará tu equipo.

In [ ]:
LOGICA = r'''"""Lógica del demostrador: cargar el modelo, validar la entrada y predecir. Sin interfaz."""
import json
import os

import joblib
import pandas as pd

CARPETA = os.path.dirname(os.path.abspath(__file__))


def cargar_modelo(carpeta=None):
    """Devuelve (modelo, metadatos). Lanza FileNotFoundError con un mensaje claro si faltan los archivos."""
    carpeta = carpeta or os.path.join(CARPETA, "modelo")
    ruta_modelo = os.path.join(carpeta, "modelo_cancelaciones.joblib")
    ruta_meta = os.path.join(carpeta, "metadatos.json")
    if not (os.path.exists(ruta_modelo) and os.path.exists(ruta_meta)):
        raise FileNotFoundError("Falta el modelo. Ejecuta primero el notebook S16_01 (o tu script de entrenamiento) para crear la carpeta 'modelo/'.")
    with open(ruta_meta, encoding="utf8") as f:
        return joblib.load(ruta_modelo), json.load(f)


def validar_reserva(reserva, meta):
    """Devuelve una lista de errores en español (lista vacía = entrada válida)."""
    errores = []
    for campo, regla in meta["campos"].items():
        if campo not in reserva or reserva[campo] in (None, ""):
            errores.append(f"Falta el campo '{campo}'.")
            continue
        valor = reserva[campo]
        if regla["tipo"] == "categoria":
            if valor not in regla["valores"]:
                errores.append(f"'{valor}' no es un valor válido para '{campo}'. Opciones: {', '.join(regla['valores'])}.")
        else:
            try:
                numero = float(valor)
            except (TypeError, ValueError):
                errores.append(f"'{campo}' debe ser un número (has escrito '{valor}').")
                continue
            if not regla["min"] <= numero <= regla["max"]:
                errores.append(f"'{campo}' debe estar entre {regla['min']} y {regla['max']} (has escrito {valor}).")
    return errores


def nivel_riesgo(probabilidad):
    """Traduce la probabilidad a una etiqueta que entiende quien no sabe de modelos."""
    if probabilidad < 0.30:
        return "bajo"
    if probabilidad < 0.50:
        return "medio"
    return "alto"


def predecir_reserva(reserva, modelo, meta):
    """Valida y predice. Devuelve un diccionario; si hay errores, solo {'errores': [...]}."""
    errores = validar_reserva(reserva, meta)
    if errores:
        return {"errores": errores}
    fila = pd.DataFrame([reserva])[list(meta["campos"])]
    probabilidad = float(modelo.predict_proba(fila)[0, 1])
    return {
        "errores": [],
        "probabilidad": round(probabilidad, 3),
        "nivel": nivel_riesgo(probabilidad),
        "tasa_base": meta["tasa_cancelacion_base"],
        "aviso": (f"Estimación orientativa. El modelo tiene un AUC de {meta['auc_test']} en datos de prueba: "
                  "se equivoca con frecuencia. "
                  + meta["limitaciones"]),
    }
'''
with open(os.path.join(CARPETA, "logica.py"), "w", encoding="utf8") as f:
    f.write(LOGICA)

sys.path.insert(0, CARPETA)
import importlib, logica
importlib.reload(logica)
modelo_cargado, meta_cargado = logica.cargar_modelo()

reserva_ejemplo = {"antelacion_dias": 120, "noches": 5, "adultos": 2, "ninos": 1, "precio_noche": 95.0, "cliente_repetidor": 0,
                   "cancelaciones_previas": 0, "tipo_habitacion": "estandar", "canal": "web_intermediaria", "tarifa": "flexible", "deposito": "sin_deposito"}
print(logica.predecir_reserva(reserva_ejemplo, modelo_cargado, meta_cargado))
print(logica.predecir_reserva({**reserva_ejemplo, "noches": -3, "canal": "telepatia"}, modelo_cargado, meta_cargado))

Fíjate en tres cosas de esta lógica:
1. **Errores en español y accionables:** dicen qué campo falla, qué se esperaba y qué se recibió.
2. **El resultado incluye el aviso de incertidumbre** (AUC, límites, uso no permitido): viaja con cada respuesta.
3. **Devuelve datos, no imprime:** así la misma función sirve a la app, a los widgets y a la API.

### Ejercicio 1
Las validaciones por campo no detectan **incoherencias entre campos**. Escribe `validar_coherencia(reserva)`
que devuelva una lista de errores (vacía si todo va bien) con estas dos reglas:
1. Si hay niños (`ninos > 0`) tiene que haber al menos un adulto (`adultos >= 1`):
   error `"Los menores deben viajar con al menos un adulto."`
2. Si el importe total (`noches * precio_noche`) supera 5000 €: error que empiece por
   `"El importe total supera los 5000 €"`.

In [ ]:
def validar_coherencia(reserva):
    errores = []
    # TODO: Dos condiciones independientes con if; añade el texto exacto a la lista
    ...
    return errores

assert validar_coherencia(reserva_ejemplo) == [], "La reserva de ejemplo es coherente: la lista debe estar vacía"
assert validar_coherencia({**reserva_ejemplo, "adultos": 0, "ninos": 2}) == ["Los menores deben viajar con al menos un adulto."], "Niños sin adultos: revisa el texto del error"
err = validar_coherencia({**reserva_ejemplo, "noches": 20, "precio_noche": 400.0})
assert len(err) == 1 and err[0].startswith("El importe total supera los 5000 €"), "Un importe de 8000 € debe dar un error que empiece por 'El importe total supera los 5000 €'"
print("Ejercicio 1 correcto ✔")

### Ejercicio 2
Escribe `mensaje_errores(errores)`: recibe una lista de errores y devuelve **un único texto** para mostrar
al usuario. Debe empezar por `"No se puede calcular todavía:"` y listar cada error en una línea que empiece
por `"- "`. Si la lista está vacía, devuelve una cadena vacía `""`.

In [ ]:
def mensaje_errores(errores):
    # TODO: Si no hay errores devuelve "" ; si los hay, une con saltos de línea las líneas "- ..."
    ...

assert mensaje_errores([]) == "", "Sin errores, devuelve una cadena vacía"
m = mensaje_errores(["A", "B"])
assert m.startswith("No se puede calcular todavía:") and "- A" in m and "- B" in m, "Formato: cabecera y una línea '- error' por cada error"
print("Ejercicio 2 correcto ✔")
print(mensaje_errores(logica.validar_reserva({**reserva_ejemplo, "noches": -3, "canal": "telepatia"}, meta_cargado)))

## 3. Pruebas de sentido común

**Idea.** Un modelo puede fallar sin dar error: devolver probabilidades absurdas. Antes de enseñarlo,
escribimos **pruebas** (`assert`) sobre casos cuyo resultado esperamos: *"una reserva con riesgo alto sobre
el papel debe puntuar más que una de riesgo bajo"*. Estas pruebas se guardan en el repositorio
(`tests/test_logica.py`) y suman puntos en el criterio F (pruebas).

### Ejercicio 3
Define dos reservas completas (diccionarios con los 11 campos, como `reserva_ejemplo`):
`reserva_riesgo_alto` (mucha antelación, sin depósito, tarifa flexible, canal `web_intermediaria`) y
`reserva_riesgo_bajo` (poca antelación, depósito total, tarifa no reembolsable, canal `directo`).
Calcula sus probabilidades en `p_alto` y `p_bajo` con `logica.predecir_reserva`.

In [ ]:
# TODO: Copia reserva_ejemplo con {**reserva_ejemplo, "campo": valor} y cambia solo lo necesario; la probabilidad está en la clave "probabilidad"
...

assert logica.validar_reserva(reserva_riesgo_alto, meta_cargado) == [] and logica.validar_reserva(reserva_riesgo_bajo, meta_cargado) == [], "Ambas reservas deben ser válidas (11 campos, valores permitidos)"
assert 0 <= p_bajo <= 1 and 0 <= p_alto <= 1, "Las probabilidades deben estar entre 0 y 1"
assert p_alto > p_bajo + 0.3, "La reserva de riesgo alto debería superar a la de riesgo bajo en más de 0,30; revisa qué campos has cambiado"
print(f"Ejercicio 3 correcto ✔ — riesgo alto {p_alto:.0%} · riesgo bajo {p_bajo:.0%}")

In [ ]:
# Estas pruebas se guardarían tal cual en tests/test_logica.py (se ejecutan con:  pytest -q)
def test_reserva_valida_devuelve_probabilidad():
    r = logica.predecir_reserva(reserva_ejemplo, modelo_cargado, meta_cargado)
    assert 0 <= r["probabilidad"] <= 1 and r["nivel"] in ("bajo", "medio", "alto") and "aviso" in r

def test_entrada_invalida_no_rompe():
    r = logica.predecir_reserva({**reserva_ejemplo, "noches": -3}, modelo_cargado, meta_cargado)
    assert r["errores"] and "probabilidad" not in r

def test_falta_un_campo():
    incompleta = {k: v for k, v in reserva_ejemplo.items() if k != "canal"}
    assert any("canal" in e for e in logica.validar_reserva(incompleta, meta_cargado))

def test_sentido_comun():
    assert p_alto > p_bajo

for prueba in (test_reserva_valida_devuelve_probabilidad, test_entrada_invalida_no_rompe, test_falta_un_campo, test_sentido_comun):
    prueba()
    print("OK ", prueba.__name__)

## 4. Opción A (recomendada): app de Streamlit

**Idea.** Streamlit convierte un script de Python en una web. El script se **vuelve a ejecutar de arriba
abajo cada vez que el usuario toca algo**. Solo necesita la lógica de la sección 2.

**Ejemplo resuelto:** generamos `demostrador/app.py`. **No lo ejecutamos aquí**: se lanza desde una terminal.

```bash
pip install streamlit joblib scikit-learn pandas
cd demostrador
streamlit run app.py
```
En Colab, Streamlit no se abre directamente (hace falta un túnel); por eso en Colab usa la **opción B**
(widgets) y deja Streamlit para el equipo en local o en *Streamlit Community Cloud*.

In [ ]:
APP = r'''"""Demostrador Streamlit · TurisData. Lanzar con:  streamlit run app.py"""
import streamlit as st

from logica import cargar_modelo, predecir_reserva

st.set_page_config(page_title="Riesgo de cancelación · TurisData", layout="centered")


@st.cache_resource
def cargar():
    return cargar_modelo()


st.title("Riesgo de cancelación de una reserva")
st.caption("Demostrador del proyecto. Datos sintéticos de TurisData Canarias (empresa ficticia).")

try:
    modelo, meta = cargar()
except FileNotFoundError as error:
    st.error(str(error))
    st.stop()

with st.form("formulario"):
    reserva = {}
    columnas = st.columns(2)
    for i, (campo, regla) in enumerate(meta["campos"].items()):
        with columnas[i % 2]:
            if regla["tipo"] == "categoria":
                reserva[campo] = st.selectbox(regla["etiqueta"], regla["valores"])
            elif regla["tipo"] == "entero":
                reserva[campo] = st.number_input(regla["etiqueta"], value=int(max(regla["min"], min(regla["max"], 1))), step=1)
            else:
                reserva[campo] = st.number_input(regla["etiqueta"], value=float(regla["min"] + 75.0), step=5.0)
    enviado = st.form_submit_button("Calcular riesgo")

if enviado:
    resultado = predecir_reserva(reserva, modelo, meta)
    if resultado["errores"]:
        st.warning("No se puede calcular todavía:\n\n" + "\n".join(f"- {e}" for e in resultado["errores"]))
    else:
        c1, c2 = st.columns(2)
        c1.metric("Probabilidad de cancelación", f"{resultado['probabilidad']:.0%}", f"{resultado['probabilidad'] - resultado['tasa_base']:+.0%} frente a la media")
        c2.metric("Nivel de riesgo", resultado["nivel"].upper())
        st.progress(resultado["probabilidad"])
        st.info(resultado["aviso"])
'''
with open(os.path.join(CARPETA, "app.py"), "w", encoding="utf8") as f:
    f.write(APP)
py_compile.compile(os.path.join(CARPETA, "app.py"), doraise=True)      # comprueba que no tiene errores de sintaxis
print("app.py creado y sin errores de sintaxis. Para verlo: cd demostrador && streamlit run app.py")

## 5. Opción B: notebook con widgets (plan B sin instalar nada)

Con `ipywidgets` (viene en Colab) se crea un pequeño formulario dentro del propio notebook. Sirve de
**plan B** si falla la app durante la defensa. Si ipywidgets no está instalado, la celda lo avisa y no falla.

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    HAY_WIDGETS = True
except ImportError:
    HAY_WIDGETS = False
    print("ipywidgets no está instalado. En local: pip install ipywidgets (en Colab ya viene incluido).")

if HAY_WIDGETS:
    campos = {}
    for campo, regla in meta_cargado["campos"].items():
        if regla["tipo"] == "categoria":
            campos[campo] = widgets.Dropdown(options=regla["valores"], description=regla["etiqueta"][:22], style={"description_width": "170px"})
        else:
            campos[campo] = widgets.FloatText(value=float(reserva_ejemplo[campo]), description=regla["etiqueta"][:22], style={"description_width": "170px"})
    boton = widgets.Button(description="Calcular riesgo", button_style="primary")
    salida = widgets.Output()

    def al_pulsar(_):
        with salida:
            clear_output()
            reserva = {c: (w.value if isinstance(w, widgets.Dropdown) else (int(w.value) if meta_cargado["campos"][c]["tipo"] == "entero" else w.value))
                       for c, w in campos.items()}
            r = logica.predecir_reserva(reserva, modelo_cargado, meta_cargado)
            if r["errores"]:
                print(mensaje_errores(r["errores"]))
            else:
                print(f"Probabilidad de cancelación: {r['probabilidad']:.0%}  (riesgo {r['nivel']}; media histórica {r['tasa_base']:.0%})")
                print(r["aviso"])
    boton.on_click(al_pulsar)
    display(widgets.VBox(list(campos.values()) + [boton, salida]))

## 6. Opción C (opcional): API con FastAPI

**Solo si tu proyecto lo necesita** (p. ej. otro sistema consulta tu modelo). La API expone la misma lógica
en dos rutas: `/salud` (¿está viva?) y `/predecir` (recibe una reserva en JSON). FastAPI valida los tipos
y rangos automáticamente y genera documentación interactiva en `/docs`.

```bash
pip install fastapi uvicorn
cd demostrador
uvicorn api:app --reload        # después abre http://127.0.0.1:8000/docs
```
**OPCIONAL.** Si `fastapi` no está instalado (o no hay `httpx`), la comprobación de abajo se salta.

In [ ]:
API = r'''"""API local del modelo. Lanzar con:  uvicorn api:app --reload"""
from typing import Literal

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

from logica import cargar_modelo, predecir_reserva

app = FastAPI(title="Riesgo de cancelación · TurisData", version="1.0")
modelo, meta = cargar_modelo()


class Reserva(BaseModel):
    antelacion_dias: int = Field(ge=0, le=1000)
    noches: int = Field(ge=1, le=30)
    adultos: int = Field(ge=1, le=6)
    ninos: int = Field(ge=0, le=6)
    precio_noche: float = Field(ge=20, le=600)
    cliente_repetidor: int = Field(ge=0, le=1)
    cancelaciones_previas: int = Field(ge=0, le=10)
    tipo_habitacion: str
    canal: str
    tarifa: Literal["flexible", "no_reembolsable"]
    deposito: Literal["sin_deposito", "parcial", "total"]


@app.get("/salud")
def salud():
    return {"estado": "ok", "modelo": meta["nombre"], "auc_test": meta["auc_test"]}


@app.post("/predecir")
def predecir(reserva: Reserva):
    resultado = predecir_reserva(reserva.model_dump(), modelo, meta)
    if resultado["errores"]:
        raise HTTPException(status_code=422, detail=resultado["errores"])
    return resultado
'''
with open(os.path.join(CARPETA, "api.py"), "w", encoding="utf8") as f:
    f.write(API)
py_compile.compile(os.path.join(CARPETA, "api.py"), doraise=True)
print("api.py creado y sin errores de sintaxis.")

In [ ]:
# Comprobación OPCIONAL de la API (se salta si faltan fastapi o httpx)
try:
    from fastapi.testclient import TestClient
    import api
    importlib.reload(api)
    cliente = TestClient(api.app)
    print("/salud →", cliente.get("/salud").json())
    ok = cliente.post("/predecir", json=reserva_ejemplo)
    print("/predecir (válida) →", ok.status_code, ok.json()["probabilidad"])
    mal = cliente.post("/predecir", json={**reserva_ejemplo, "noches": -3})
    print("/predecir (noches=-3) →", mal.status_code, "(la API rechaza la entrada errónea)")
    mal2 = cliente.post("/predecir", json={**reserva_ejemplo, "canal": "telepatia"})
    print("/predecir (canal inventado) →", mal2.status_code, mal2.json()["detail"][0][:60])
except ImportError as e:
    print("Comprobación de la API omitida (opcional):", e)

## 7. Lista de reproducibilidad del demostrador

Un compañero de otro equipo debería poder ejecutarlo **solo con el README**. Comprueba:
- [ ] `requirements.txt` con **versiones fijas** (`pip freeze` de las librerías que usas, no de todo el sistema).
- [ ] El modelo se crea con un **script o notebook** (`entrenar.py`) o se incluye en el repo (si pesa poco).
- [ ] El README dice **exactamente** qué comandos ejecutar y qué se verá.
- [ ] La app **no se rompe** con entradas erróneas y muestra el aviso de incertidumbre.
- [ ] Hay **pruebas** (`pytest -q`) y **plan B**: vídeo de 1-2 min de la demo funcionando.
- [ ] No hay claves de API ni datos personales en el repositorio.

In [ ]:
# Generamos un requirements.txt con las versiones exactas de este entorno (revísalo y recórtalo a lo que uses)
import importlib.metadata as md
paquetes = ["numpy", "pandas", "scikit-learn", "joblib", "streamlit", "ipywidgets", "fastapi", "uvicorn"]
lineas = []
for p in paquetes:
    try:
        lineas.append(f"{p}=={md.version(p)}")
    except md.PackageNotFoundError:
        lineas.append(f"# {p}  (no instalado aquí: fija la versión que uses)")
with open(os.path.join(CARPETA, "requirements.txt"), "w", encoding="utf8") as f:
    f.write("\n".join(lineas) + "\n")
print("\n".join(lineas))

## Mini-reto integrador
Adapta la plantilla a **tu proyecto**: (1) guarda tu mejor modelo y sus metadatos; (2) modifica `logica.py`
(campos, validaciones y aviso de incertidumbre propios); (3) lanza `app.py`; (4) escribe 3 pruebas;
(5) graba un vídeo de 90 segundos como plan B. Comprueba que otra persona del equipo lo ejecuta desde cero.

## Preguntas de reflexión
1. ¿Por qué es mejor que `logica.py` no importe Streamlit? ¿Qué ganamos al reutilizarla en la API y en el notebook?
2. El aviso de incertidumbre viaja en cada respuesta. ¿Qué otro aviso añadirías para tu proyecto (imágenes, texto, series)?
3. ¿Qué pasa si el usuario introduce un valor fuera del rango de entrenamiento y la validación lo permite? ¿Cómo lo detectarías?
4. ¿Qué datos personales podría recibir tu demostrador y cómo evitas guardarlos?

## Qué has aprendido
- Guardar modelo + metadatos y cargarlos desde otra capa.
- Separar lógica e interfaz; validar entradas y mostrar incertidumbre.
- Escribir pruebas de sentido común para un modelo.
- Tres formas de demostrador (Streamlit, widgets, API) y el plan B.